# Prepare cache

By default (`paths.sharedCacheRoot` in `configs/base.yaml`) this PNPP model reuses the RandLA-Net cache (read only), so both reference models are trained and evaluated on exactly the same points, labels, `u_tilde`, PAvPU patches and PAvPU threshold. In that case this notebook only checks that the cache is there and prints its contents. Nothing is written into the RandLA-Net folder

## Setup

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

from src import paths
print('project folder:', paths.PROJECT_ROOT)

from src.config import loadConfig
from src.prepare import loadManifest, prepareDataset

In [ ]:
DATASETS  = ['paris_lille', 'toronto3d', 'ahn3','vaihingen']
RUN_TESTS = True

## Tests on synthetic data

In [ ]:
if RUN_TESTS:
    import contextlib, importlib, io
    sys.path.insert(0, os.path.join(ROOT, 'tests'))
    import test_numpy_parts as T
    importlib.reload(T)
    tests = [getattr(T, n) for n in dir(T) if n.startswith('test') and callable(getattr(T, n))]
    for t in tests:
        with contextlib.redirect_stdout(io.StringIO()):
            t()
        print(f'ok  {t.__name__}')
    print(f'{len(tests)} tests passed')

## Build or check the cache
Datasets that fail are reported and skipped.

In [ ]:
manifests, errors = {}, []
for ds in DATASETS:
    print(f'=== {ds} ===')
    try:
        cfg = loadConfig(ds)
        if paths.usingSharedCache(cfg):
            manifests[ds] = loadManifest(cfg)
            print(f'using the shared RandLA-Net cache (read only): {paths.cacheDir(cfg)}')
        else:
            manifests[ds] = prepareDataset(cfg)
    except Exception as e:
        print(f'[{ds}] FAILED: {e}')
        errors.append({'dataset': ds, 'error': str(e)})
    print()
if errors:
    display(pd.DataFrame(errors))

## Summary

In [ ]:
rows = []
for ds, m in manifests.items():
    for split, info in (('train', m['trainInfo']), ('test', m['testInfo'])):
        for f, i in info.items():
            rows.append({'dataset': ds, 'split': split, 'file': f, 'points': i['n'],
                         'extent_x_m': i['extent_m'][0], 'extent_y_m': i['extent_m'][1],
                         'extent_z_m': i['extent_m'][2], 'u_max': round(i['uMax'], 4)})
display(pd.DataFrame(rows))
print('Extents should be in metres (hundreds of metres for a tile). Values near 1 mean the input is not metric.')